# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 '

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwar

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog / posts page', 'url': 'https://edwarddonner.com/posts/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog / news', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'partner / related company',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
Reachy Mini: The Open Robot for AI Builders
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
PaddlePaddle/PaddleOCR-VL
Updated
1 day ago
•
3.82k
•
638
nanonets/Nanonets-OCR2-3B
Updated
4 days ago
•
12.8k
•
323
Qwen/Qwen3-VL-8B-Instruct
Updated
4 days ago
•
74.5k
•
198
Phr00t/Qwen-Image-Edit-Rapid-AIO
Updated
1 day ago
•
356
inclusionAI/Ring-1T
Updated
6 days ago
•
435
•
175
Browse 1M+ models
Spaces
Running
325
325
veo3.1-fast
🐨
Generate videos from text or images
Running
15.2k
15.2k
DeepSite v3
🐳
Gene

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nTry HuggingChat Omni – Chat with AI 💬\nGet started with Inference in seconds 🚀\nReachy Mini: The Open Robot for AI Builders\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nPaddlePaddle/PaddleOCR-VL\nUpdated\nabout 12 hours ago\n•\n3.07k\n•\n554\nnanonets/Nanonets-OCR2-3B\nUpdated\n3 days ago\n•\n11.1k\n•\n300\ninclusionAI/Ling-1T\nUpdated\n5 days ago\n•\n2.89k\n•\n447\nPhr00t/Qwen-Image-Edit-Rapid-AIO\nUpdated\n32 minutes ago\n•\n3

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future — a vibrant collaboration platform for machine learning (ML) enthusiasts, engineers, scientists, and enterprises worldwide. It serves as the central hub where anyone can share, explore, and experiment with open-source machine learning models, datasets, and applications. Hugging Face empowers the global ML community to build an open, ethical, and innovative AI future together.

---

## What We Offer

- **Models:** Access and browse over **1 million ML models** spanning text, image, video, audio, and even 3D modalities.
- **Datasets:** Explore a rich collection of over **250,000 datasets**, facilitating easier training and benchmarking for ML projects.
- **Spaces:** Host and interact with **over 400,000 AI applications**, enabling users to deploy, showcase, and collaborate on live ML apps.
- **Open Source Tools:** Utilize the Hugging Face open-source stack to accelerate your ML development.
- **Enterprise Solutions:** Benefit from paid compute resources and advanced platforms tailored for team and enterprise needs.
- **HuggingChat Omni:** Experience conversational AI via their latest chatbot platform.

---

## The Hugging Face Community

Hugging Face is more than a platform — it’s a thriving global community committed to the future of machine learning. By fostering collaboration across researchers, developers, and users, the community advances the boundaries of AI through:

- Sharing and jointly improving open-source projects.
- Hosting discussions and knowledge exchange.
- Building portfolios and reputations within the community.
  
Join thousands of contributors who are continuously pushing the envelope in ML innovation.

---

## Company Culture

At Hugging Face, the culture centers around:

- **Openness:** A commitment to open-source principles and transparent collaboration.
- **Ethics:** Building and advocating for responsible AI use.
- **Innovation:** Empowering the next generation of AI talent and projects.
- **Community:** Inclusive, supportive, and passionate about shared progress.
  
Hugging Face nurtures talent that is proactive, curious, and values collaboration, making it a magnet for passionate AI professionals.

---

## Careers & Opportunities

Hugging Face is actively expanding its team in areas such as:

- Machine Learning Engineering
- Research Science
- Software Development
- Product Management
- Customer & Developer Support
  
By joining Hugging Face, you become part of a mission-driven company shaping the future of AI through open collaboration. The company encourages applicants who are eager to learn, develop cutting-edge ML technologies, and contribute to an ethical AI ecosystem.

---

## Who Uses Hugging Face?

Hugging Face serves:

- Individual developers and researchers exploring or deploying ML models.
- AI builders creating applications across various data modalities.
- Enterprises needing scalable and secure ML infrastructure.
- Academic institutions advancing AI research.
  
With tools and resources tailored for beginners to experts, Hugging Face supports a broad spectrum of customers working on real-world AI challenges.

---

## Connect & Explore

- **Website:** [huggingface.co](https://huggingface.co)  
- **Browse Models:** Explore 1M+ machine learning models  
- **Datasets:** Access 250k+ datasets ready for use  
- **Spaces:** Discover and deploy AI applications  
- **Community:** Join discussions, collaborations, and knowledge sharing  

---

### Hugging Face — The Home of Machine Learning Collaboration

Empower your AI journey with the most advanced open-source platform designed to accelerate innovation through collaboration, openness, and community spirit. Join Hugging Face today and be a part of shaping the future of AI.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is at the forefront of building the future of Artificial Intelligence through a vibrant, open, and collaborative machine learning community. It's a dynamic platform where machine learning engineers, scientists, and AI enthusiasts come together to create, share, explore, and experiment with state-of-the-art models, datasets, and AI applications. 

As the home of machine learning, Hugging Face offers an expansive ecosystem featuring:

- Over **1 million models**
- More than **250,000 datasets**
- **400,000+ AI applications**

Their platform supports all AI modalities including text, image, video, audio, and 3D, making it the place to accelerate machine learning for all users.

---

## What Makes Hugging Face Unique?

- **Collaboration at the Core:** Hugging Face Hub enables seamless hosting, sharing, and collaborative work on public models, datasets, and applications.
- **Open Source Innovation:** The Hugging Face Open Source stack empowers developers to move faster with cutting-edge machine learning libraries and tools.
- **Diverse AI Modalities:** Explore and build solutions across text, images, video, audio, and 3D data.
- **Community-Driven Growth:** The platform thrives on its fast-growing community where contributors share knowledge, best practices, and innovations.
- **Enterprise & Compute Solutions:** Beyond community resources, Hugging Face offers paid compute power and specialized enterprise platforms to meet business needs.

---

## Customers & Community

Hugging Face serves a global base of machine learning enthusiasts, researchers, startups, and enterprises. The community consists of:

- Independent AI builders and researchers
- Educational institutions leveraging vast datasets and models
- Large enterprises deploying AI at scale using Hugging Face Enterprise tools
- Developers using thousands of open-source models for innovation
- AI engineers building portfolios and showcasing their work publicly

The platform also features popular AI applications and tools such as **HuggingChat Omni** for AI conversational experiences, and Reachy Mini, an open robot designed for AI developers.

---

## Company Culture

Hugging Face fosters an open, inclusive, and ethical AI culture. They empower the next generation of AI professionals by encouraging:

- Openness and transparency through shared resources
- Collaboration across disciplines and borders
- Ethical AI development aligned with community values
- Continuous learning and growth through shared portfolios and community interaction
- Innovation driven by user feedback and open-source contributions

---

## Careers at Hugging Face

Hugging Face is continually looking for passionate individuals who want to contribute to the future of AI. Careers here offer opportunities to:

- Work with cutting-edge machine learning technologies in a collaborative environment
- Contribute directly to open-source tools impacting millions worldwide
- Engage with a global, engaged AI community
- Build and scale enterprise AI solutions alongside industry experts

Whether you are a researcher, engineer, or community builder, Hugging Face offers a space to grow your skills and impact the AI landscape.

---

## Get Started

Join the AI revolution at Hugging Face:

- Explore over **1 million public models**
- Collaborate on datasets and cutting-edge AI apps
- Build your own machine learning portfolio
- Try **HuggingChat Omni** for AI conversations

Discover, create, and innovate at [huggingface.co](https://huggingface.co).

---

### Brand Colors & Assets

- Primary Yellow: #FFD21E
- Accent Orange: #FF9D00
- Neutral Gray: #6B7280

Official logos and brand assets are available for partners and collaborators.

---

Hugging Face – **The AI community building the future.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It is a leading platform where machine learning engineers, scientists, and AI builders from around the world collaborate to create, share, and innovate open-source models, datasets, and applications. 

Hugging Face empowers its users to learn, experiment, and contribute to machine learning in an open and ethical way, making it a home for the global AI community.

---

## What Hugging Face Offers

- **Model Hub:** Access and browse over 1 million pre-trained models across multiple modalities including text, image, video, audio, and 3D. These models are regularly updated and publicly available for experimentation and deployment.

- **Datasets:** Explore 250,000+ datasets curated for AI tasks, enabling smoother and more effective training and benchmarking.

- **Spaces:** A growing collection of over 400,000 AI-powered applications and demos created by the community, facilitating innovation through sharing.

- **HuggingChat Omni:** An interactive chat platform to engage with AI models directly.

- **Open Source Stack:** A robust open-source infrastructure that accelerates machine learning efforts and pushes boundaries in AI development.

- **Compute and Enterprise Solutions:** Paid plans to enable teams and organizations with scalable compute resources and the most advanced collaboration tools. 

---

## Community & Collaboration

Hugging Face prides itself on being more than just a platform—it is a thriving, fast-growing community that supports:

- Wide access to open-source ML libraries and models.
- Opportunities for AI enthusiasts and professionals to build their portfolios and advance their skills.
- Cross-disciplinary collaboration between researchers, developers, and industry experts.
  
The platform fuels innovation through transparency, sharing, and ethical AI development practices.

---

## Company Culture

- **Open & Inclusive:** Dedicated to building an open AI community accessible to anyone passionate about machine learning.
- **Collaborative:** Encourages teamwork and collective progress in AI through open sharing and partnerships.
- **Innovative:** Constantly pushing frontiers with new AI tools and solutions tailored to diverse modalities.
- **Ethical AI:** Committed to developing AI technologies responsibly with a focus on inclusivity and societal benefit.

---

## Careers & Opportunities

Hugging Face offers exciting career opportunities for machine learning engineers, data scientists, software developers, AI researchers, and more. Joining Hugging Face means being part of a mission-driven company that values innovation, transparency, and community impact.

- Work alongside a global community of AI pioneers.
- Contribute to bleeding-edge AI open-source projects and products.
- Access resources that allow rapid prototyping and production of ML solutions.

Explore current openings on their website and become a part of shaping the future of AI.

---

## Customers & Impact

Hugging Face serves a diverse, global audience including:

- Individual developers and AI enthusiasts building and sharing models.
- Academic researchers pushing the boundaries of AI science.
- Industry teams leveraging enterprise-grade ML infrastructure.
- Companies seeking scalable, ethical, and open AI solutions.

With hundreds of thousands of active users and contributors, Hugging Face is a foundational pillar in the AI ecosystem.

---

## Get Started

Visit [huggingface.co](https://huggingface.co) to:

- Browse models and datasets
- Join the community
- Sign up to create your AI apps and portfolios
- Discover enterprise solutions for your team or business

Join Hugging Face — the collaboration platform accelerating the future of machine learning.

---

*Hugging Face*  
**Colors:** Bright Yellow (#FFD21E), Orange (#FF9D00), and Gray (#6B7280)  
**Tagline:** The AI community building the future.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>